# 01 · Caso de negocio · Venta cruzada de seguros

> **BORRADOR.** Escrito a partir de los datos de entrenamiento (`data/raw/train.csv` o `dataset.csv`). Cada decisión está marcada **[DECIDIDO]** (resuelto con evidencia del proyecto), **[ADOPTADO]** (propuesta vigente por defecto hasta que el grupo diga lo contrario) o **[PENDIENTE: dato externo]** (no se puede resolver con estos datos); el registro completo está en `reports/decisiones_del_grupo.md`. Lo pendiente es una hipótesis que el rol
> *Negocio* debe confirmar o cambiar. Ejemplo completo: `material-del-curso/ejemplo-creditsight/notebooks/01_caso_negocio.ipynb`.

Antes de abrir un solo dato, el proyecto necesita **una pregunta de negocio, una métrica que la refleje y un costo de equivocarse**.
Este notebook termina con esas tres cosas escritas **con números**, y su resumen va a `reports/00_caso.md`.

**Quién lo escribe:** el rol *Negocio* del grupo, con los demás revisando.

## 1. La pregunta de negocio

**Pregunta de negocio:** ¿a qué clientes que ya tienen seguro de salud conviene ofrecerles también un seguro de vehículo?
**[PENDIENTE: dato externo · P06]**

**¿Quién decide y con qué información?** El equipo comercial de la aseguradora decide a quién llama primero. Dispone de la edad, el género,
si el cliente ya tiene seguro de vehículo, la antigüedad del vehículo, si tuvo daños, la prima anual, la región y el canal de venta.

**¿Qué pasa hoy, sin el modelo?** **[PENDIENTE: dato externo · P06]** Sin modelo se llama a clientes sin orden o con una regla simple. Como solo 1 de cada 8 acepta,
la mayor parte de las llamadas se pierde.

## 2. Equivocarse no cuesta lo mismo

Hay dos errores posibles al decidir a quién llamar:

- **Error más caro: no llamar a quien habría comprado** (falso negativo). Es una venta perdida y la aseguradora no se entera.
- **Error menos caro: llamar a quien no comprará** (falsa alarma). Cuesta el tiempo de una llamada.

**Proporción entre los dos:** se usa **5 a 1** como *hipótesis de trabajo*. **[PENDIENTE: dato externo · P01]** No sale de los datos (y **las etapas 5 y 6 mostraron que los finalistas y el umbral no cambian con R entre 2 y 12**, así que no bloquea el avance): debe salir de comparar
el margen de una venta con el costo de una llamada. Mientras no se confirme, la constante `COST_FALSE_NEGATIVE` de la celda siguiente
se puede cambiar y todo lo demás se recalcula.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Funciona si el notebook se ejecuta desde notebooks/ o desde la raíz del repo
ROOT = Path.cwd().resolve()
ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from proyecto.data.cleaning import find_train_csv

# Busca data/raw/train.csv, data/raw/dataset.csv o dataset.csv en la raíz del repo
DATA = find_train_csv(ROOT)
df = pd.read_csv(DATA, usecols=["Response", "Vehicle_Damage", "Previously_Insured"])
print(f"{len(df):,} clientes leídos de {DATA.relative_to(ROOT)}")

CLASSES = ["no acepta", "acepta"]

# Hipótesis de trabajo [PENDIENTE: dato externo · P01]: no llamar a un comprador cuesta 5 veces más que una llamada inútil.
COST_FALSE_NEGATIVE = 5
COST_FALSE_POSITIVE = 1

# Filas = lo que realmente pasó, columnas = lo que predijo el modelo. Acertar no cuesta.
COST_MATRIX = [[0, COST_FALSE_POSITIVE],
               [COST_FALSE_NEGATIVE, 0]]

cost = pd.DataFrame(COST_MATRIX, index=[f"real {c}" for c in CLASSES], columns=[f"predice {c}" for c in CLASSES])
display(cost)

381,109 clientes leídos de data\raw\train.csv


,predice no acepta,predice acepta
real no acepta,0,1
real acepta,5,0


### La métrica de costo: cuánto cuesta equivocarse en cada caso

Se mide todo en **"unidades de llamada"**: **1 unidad = lo que cuesta una llamada que no termina en venta.** Así no hace falta inventar dólares.

| Caso | Qué pasó | Costo | Cómo se calcula |
|---|---|---|---|
| **Acierto: contactar a un comprador** | Llamamos y compra | 0 | Se gana la venta; la llamada se paga con ella |
| **Acierto: no llamar a quien no compra** | No llamamos y no habría comprado | 0 | No se gasta nada |
| **Falsa alarma (falso positivo)** | Llamamos a quien no compra | **1** | El costo de una llamada: `COST_FALSE_POSITIVE` |
| **Venta perdida (falso negativo)** | No llamamos a quien habría comprado | **R** | El margen de la venta que no se hizo: `COST_FALSE_NEGATIVE` |

**R = margen de una venta ÷ costo de una llamada.** Si una venta deja 5 veces lo que cuesta llamar, R = 5. El grupo debe sacar R de datos reales de la aseguradora
**[PENDIENTE: dato externo · P01]**; mientras tanto se usa 5 como hipótesis.

**Métrica:** `costo promedio por cliente = (falsas alarmas × 1 + ventas perdidas × R) ÷ número de clientes`. Menor es mejor y se puede comparar entre modelos.

**Regla de decisión que se deduce:** conviene llamar a un cliente cuando su probabilidad de comprar es mayor que **1 ÷ (R + 1)**. Con R = 5, el umbral es 16,7 %.

In [2]:
def cost_per_client(y_true, y_pred, fn_cost=COST_FALSE_NEGATIVE, fp_cost=COST_FALSE_POSITIVE):
    """Costo promedio por cliente, en unidades de llamada. y_true y y_pred son 0/1."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    false_alarms = ((y_pred == 1) & (y_true == 0)).sum()
    lost_sales = ((y_pred == 0) & (y_true == 1)).sum()
    return (false_alarms * fp_cost + lost_sales * fn_cost) / len(y_true)

y_all = df["Response"].to_numpy()
damage = (df["Vehicle_Damage"] == "Yes").astype(int).to_numpy()
p = y_all.mean()

# ¿Cómo cambian las conclusiones si R es otro? (sensibilidad)
rows = []
for R in [1, 2, 5, 10, 20]:
    rows.append({
        "R (venta perdida / llamada)": R,
        "umbral de probabilidad": 1 / (R + 1),
        "no llamar a nadie": cost_per_client(y_all, np.zeros_like(y_all), fn_cost=R),
        "llamar a todos": cost_per_client(y_all, np.ones_like(y_all), fn_cost=R),
        "regla del daño": cost_per_client(y_all, damage, fn_cost=R),
    })
sens = pd.DataFrame(rows).set_index("R (venta perdida / llamada)")
sens["mejor sin modelo"] = sens[["no llamar a nadie", "llamar a todos", "regla del daño"]].idxmin(axis=1)
display(sens.round(3))

print(f"Llamar a todos sale más barato que no llamar a nadie cuando R > {(1 - p) / p:.1f}")

,umbral de probabilidad,no llamar a nadie,llamar a todos,regla del daño,mejor sin modelo
R (venta perdida / llamada),,,,,
1,0.500,0.123,0.877,0.387,no llamar a nadie
2,0.333,0.245,0.877,0.390,no llamar a nadie
5,0.167,0.613,0.877,0.398,regla del daño
10,0.091,1.226,0.877,0.411,regla del daño
20,0.048,2.451,0.877,0.436,regla del daño


Llamar a todos sale más barato que no llamar a nadie cuando R > 7.2


**Cómo leer la tabla:** cuanto más vale una venta frente a una llamada (R grande), más conviene llamar a mucha gente y más bajo es el umbral de probabilidad.
Con R chico, conviene ser selectivo. Con R = 1 las dos equivocaciones cuestan igual. **Esta tabla es lo que el grupo lleva al negocio** para acordar R.

## 3. ¿Contra qué nos comparamos? (línea base)

Una métrica sola no dice nada: necesita un punto de referencia. Se calcula el costo promedio por cliente de **tres estrategias sin modelo**:
- **Siempre "no acepta"** (no llamar a nadie).
- **Siempre "acepta"** (llamar a todos).
- **Azar según las proporciones.**

Y una cuarta, una **regla simple** de negocio: llamar solo a quien tuvo daño en el vehículo.

In [3]:
y = df["Response"].to_numpy()
priors = np.array([(y == 0).mean(), (y == 1).mean()])   # proporciones reales de "no acepta" y "acepta"
cm = np.array(COST_MATRIX, dtype=float)

print(f"Proporciones: no acepta = {priors[0]:.2%}, acepta = {priors[1]:.2%}")

def expected_cost(pred):
    """Costo promedio por cliente si se predice `pred` (0/1) para cada uno."""
    return cm[y, pred].mean()

always_no = expected_cost(np.zeros_like(y))
always_yes = expected_cost(np.ones_like(y))
random_by_priors = priors @ cm @ priors
rule_damage = expected_cost((df["Vehicle_Damage"] == "Yes").astype(int).to_numpy())

summary = pd.DataFrame({
    "costo promedio por cliente": [always_no, always_yes, random_by_priors, rule_damage],
}, index=["Siempre 'no acepta'", "Siempre 'acepta' (llamar a todos)", "Azar según proporciones", "Regla: llamar si hubo daño"])
display(summary.round(3))

accuracy_always_no = (y == 0).mean()
print(f"\nExactitud de 'siempre no acepta': {accuracy_always_no:.2%}  (y no encuentra a ningún comprador)")

# De cada 100 compradores, ¿cuántos encuentra la regla del daño?
rule = (df["Vehicle_Damage"] == "Yes").to_numpy()
print(f"La regla del daño llama a {rule.mean():.1%} de los clientes y encuentra al {rule[y == 1].mean():.1%} de los compradores.")

Proporciones: no acepta = 87.74%, acepta = 12.26%


,costo promedio por cliente
Siempre 'no acepta',0.613
Siempre 'acepta' (llamar a todos),0.877
Azar según proporciones,0.645
Regla: llamar si hubo daño,0.398



Exactitud de 'siempre no acepta': 87.74%  (y no encuentra a ningún comprador)
La regla del daño llama a 50.5% de los clientes y encuentra al 97.9% de los compradores.


**Lo que dice la línea base (una frase):** con la proporción 5 a 1, "no llamar a nadie" cuesta menos que llamar a todos, pero no vende nada;
la regla del daño ya encuentra a casi todos los compradores llamando a la mitad de los clientes. **El modelo tiene que superar a esa regla**, no solo a "siempre no".

**¿La exactitud (accuracy) sirve como métrica en tu caso? ¿Por qué sí o por qué no?** **No sirve.** "Siempre no acepta" acierta el 87,7 % y no encuentra
ni un comprador. Con una clase tan rara, la exactitud premia al modelo que no hace nada.

## 4. Cómo se medirá el éxito

**Métrica principal: PR-AUC** (área bajo la curva precisión–recall). Mide qué tan arriba de la lista quedan los pocos compradores y no se
infla con la clase mayoritaria, como sí le pasa a la exactitud. Un modelo al azar obtiene una PR-AUC igual a la proporción de compradores
(**0,1226**), así que ese es el piso. **[DECIDIDO · D01]**

| Tipo | Métrica | Qué responde en lenguaje de negocio | Referencia para comparar |
|---|---|---|---|
| **Principal** | PR-AUC | ¿Los compradores quedan arriba en la lista de llamadas? | Azar: 0,1226 |
| **Operativa** | Recall y *lift* en el 20 % de clientes mejor puntuados | Si solo llamamos al 20 % de la lista, ¿qué parte de los compradores encontramos y cuántas veces mejor que al azar? | Azar: recall 20 %, lift 1,0 |
| **Económica** | Costo promedio por cliente con la matriz de la sección 2, al umbral elegido en validación | ¿Cuánto cuestan en promedio las decisiones de contacto? | Regla del daño: 0,398; no llamar a nadie: 0,613 |
| **Control** | ROC-AUC | ¿Ordena bien en general? Sirve para detectar problemas, no para decidir | Azar: 0,5 |
| **Control** | Brier o curva de calibración | Si el modelo dice "30 %", ¿compra realmente el 30 %? Importa porque el umbral se fija con probabilidades | Menor es mejor |
| **Equidad** | Recall y tasa de contacto por `Gender` y por tramo de `Age` | ¿Se deja de ofrecer el producto a algún grupo? | Sin diferencias grandes entre grupos |

**Qué resultado haría que el negocio diga "sirve" (propuesta, con la referencia medida):** **[DECIDIDO · D03]** **Cumplido en validación cruzada** (etapa 5: LightGBM cuesta 0,3245 por cliente frente a 0,398 de la regla simple, **18,5 % menos**); se confirma con la prueba cerrada en la etapa 7.
1. **Supera a la regla simple:** costo promedio por cliente menor que **0,398** (la regla "llamar si hubo daño"), con la misma matriz de costos. Esa regla ya
   encuentra al 97,9 % de los compradores llamando al 50,5 % de los clientes, así que ganarle exige **llamar a menos gente o acertar mejor**, no solo encontrar compradores.
2. **Concentra a los compradores:** el recall en el 20 % mejor puntuado debe quedar claramente por encima del 20 % del azar. El grupo fija la meta numérica
   cuando vea el primer modelo; no se inventa antes de medir.
3. **No discrimina:** ninguna diferencia grande de recall entre géneros ni tramos de edad.

**Reglas de uso (valen para todas las etapas):**
- El **umbral de contacto** se elige en validación minimizando el costo esperado, **nunca** en prueba.
- El conjunto de prueba se abre **una sola vez**, al final, para confirmar el resultado (`CLAUDE.md`, regla 4).
- Toda comparación entre modelos usa la **misma partición** estratificada por `Response`, para que las métricas sean comparables.
- Si la proporción de costos de la sección 2 cambia, cambian el umbral y el costo, **no** la PR-AUC. Por eso la PR-AUC es la métrica para *elegir modelo* y el costo para *elegir umbral*.

## 5. Lo que podría salir mal

- **Fuga de información (¿qué columna se conoce solo después de la decisión?):** en el EDA no se vio ninguna evidente. `Previously_Insured` y
  `Vehicle_Damage` son fuertes pero se conocen antes de llamar. `Vintage` no muestra relación con el objetivo y se medirá con y sin ella.
- **Sesgos o equidad (¿a quién podría perjudicar el modelo?):** `Gender` y `Age`. Un modelo que aprenda que cierto género o rango de edad
  "compra menos" podría dejar de ofrecer el producto a esas personas. Se revisa el desempeño por grupo en la validación.
- **Datos personales o confidenciales:** el dataset no trae nombres ni documentos; solo un `id` anónimo.
- **Hasta dónde puede decidir el modelo y dónde debe decidir una persona:** el modelo **ordena** a los clientes por probabilidad de aceptar;
  la decisión de a cuántos llamar y la forma de ofrecer el producto son del equipo comercial. El modelo no explica por qué compran ni mide el efecto de llamar.

## 6. Cierre

- [ ] Pregunta, quién decide y costos escritos **con números** (la proporción 5 a 1 sigue por confirmar).
- [ ] Línea base calculada.
- [ ] Métrica principal elegida y justificada.
- [ ] Resumen copiado a `reports/00_caso.md`.
- [ ] Alguien que **no** escribió este notebook sabe explicar por qué un error cuesta más que el otro.